# **Scipy**

## Overview

Within this notebook, we will cover:
1. How to **filter** continuous SLP data to identify features of interest.
1. How to **identify** the lowest SLP minima associated with ETCs.
2. How to use Scipy's `label` function to identify coherent objects within the SLP field.
1. How to **track** identified ETCs through successive time steps.

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Intro to Cartopy](https://foundations.projectpythia.org/core/cartopy/cartopy) | Helpful | |
| [Understanding of NetCDF](https://foundations.projectpythia.org/core/data-formats/netcdf-cf) | Necessary | Familiarity with metadata structure |
| [Intro to Numpy](https://foundations.projectpythia.org/core/numpy/) | Necessary | Basic n-dimensional array indexing

## Imports

In [ ]:
import xarray as xr
import numpy as np
from scipy.ndimage import minimum_filter
from scipy.ndimage import label, generate_binary_structure
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import math

## Core functions for this notebook

### Distance function

This function serves to calculate the distance between two given latitudes, and longitudes in **degrees**.

In [ ]:
def distance(lon1, lat1, lon2, lat2):
        radius = 6371
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2) * math.sin(dlon/2)
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
        d = radius * c
        return d


### Crosses dateline function

This function ensures that features adjecent to each other, but on the international dateline, are considered as the same feature.

In [ ]:
def crosses_dateline(labeled_arr, resolution):
    #Shift 180 degrees
    shifted_arr = np.roll(labeled_arr, 180*resolution, axis = 1)

    #Examine a 3x3 region centered at the dateline
    for lat in range(0, len(shifted_arr) - 1):
    	sample_arr = shifted_arr[lat:lat+2, 179:181]

    	top_left = sample_arr[0, 0]
    	top_right = sample_arr[0, 1]
    	bottom_right = sample_arr[1, 1]
    	bottom_left = sample_arr[1, 0]
		
    	#Compare top left with position next to it
    	if(top_left != 0 and top_right != 0):
    		if(top_left != top_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]

    	#Compare top left with position diagonal to it
    	if(top_left != 0 and bottom_right != 0):
    		if(top_left != bottom_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
		
    	#Compate bottom left with position next to it
    	if(bottom_left != 0 and bottom_right != 0):
    		if(bottom_left != bottom_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    			#print('Changed arr bot left, bot right', sample_arr)
    	
    	#Compare bottom left with position diagonal to it
    	if(bottom_left != 0 and top_right != 0):
    		if(bottom_left != top_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    		
    return np.roll(shifted_arr, 180, axis = 1)
    
    

## Read in and visualize our raw data

In [ ]:
ds = xr.open_dataset('../notebooks/data/SLP_ex.nc')
SLP = ds['SLP']
SLP_hPa = SLP / 100
lon = SLP_hPa['longitude'].values
lat = SLP_hPa['latitude'].values
lon2d, lat2d = np.meshgrid(lon, lat)
levels = np.arange(950, 1050, 10)

### Function to plot an animation over each time step in our data

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.contour(
        lon2d, lat2d, field,
        levels=levels,
        cmap=cmap,         
        linewidths=0.9,
        transform=ccrs.PlateCarree()
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Sea Level Pressure | time index = {t}")

:::{note}
The block of code below actually sets up and creates an animation of our desired data. Similar version of the blocks of code above and below this note will appear a few other times throughout this notebook. In the block of code below, we must call `.values` in the first line because `SLP_hPa` is an **xarray dataarray**, not a **numpy array**.
:::

In [ ]:
# Create map
vals = SLP_hPa.values
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("SLP")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

## Get our desired objects

Here is where we convert our **xarray dataarray** to a **numpy array**.

In [ ]:
SLP_arr = SLP_hPa.to_numpy()

### Choose a threshold for SLP

The threshold chosen here will serve as a **dividing line** between what is and is not considered a feature of interest. In the block of code below, we will choose a threshold of `1000 hPa`. This means that all of our features of interest must be comprised of grid cells whose SLP values are `1000 hPa` or less. Other thresholds could also be used, such as a departure from a long-term mean.

In [ ]:
SLP_thresh = 1000 #Units of hPa

SLP_1000 = np.where(SLP_arr < SLP_thresh, SLP, 0)

### Create our "binary structure" 

The **binary structure** is a `boolean array` that sets the definition for our *smallest* possible feature of interest. It essentially looks at all grid cells in our numpy array and determines which neighboring grid cells can be connected to form a *coherent spatial object*. The first input into `generate_binary_structure` defines the number of dimensions over which an object can cover. Since our numpy array contains three dimensions *(time, latitude, longitude)*, our desired number of dimensions will be **two**, corresponding to *latitude and longitude*. The second input defines the number of grid cells that must be connected to each other to be considered as an object. For our case here, we will define an object minimally as one that has at least two connected grid cells.

In [ ]:
s = generate_binary_structure(2,2)

### Run the label function

Here is where we actually find our desired objects given a **2-dimensional field**. When we run the label function, there will be two outputs. The first output is a **2-dimensional array** that is the exact same size as our **input array**. However, this new array contains integer values corresponding to the feature of interest. Typically, a value of zero means that the grid cell is just part of the background (i.e. not a feature of interest). Then all grid cells with a *value of 1* correspond to a feature, all grid cells with a *value of 2* correspond to another feature, and so on. The second output of label is the integer total of distinct objects that were found in the given 2-dimensional field.

In the block of code below, we first define our grid size resolution in **degrees**. Next, we set up empty arrays that will store the output from the label for each time step in our original data file. For `labeled_arr`, we want a **3-dimensional array** to store the output *(number of time steps, number of latitudes, number of longitudes)*. Since the second output of the label is only an *integer*, a **1-dimensional array** will suffice. 

Lastly, we **loop** through each time step in our data file. For our case, this is only 13 time steps. After running the label function in the for loop, we send the output to the 'crosses_dateline' function, which ensures that objects that cross the international dateline are counted as the same rather than separate objects. We then append our correct outputs to the arrays that we defined earlier.

In [ ]:
resolution = 1 #In degrees

labeled_arr = np.zeros_like(SLP_1000)

num_features_arr = np.zeros(len(SLP_1000))

for time_step in np.arange(0, len(SLP_1000)):
    labeled_SLP_field, num_features = label(SLP_1000[time_step], s)
  
    correct_labeled_SLP_field = crosses_dateline(labeled_SLP_field, resolution)
    
    num_features_arr[time_step] = num_features

    labeled_arr[time_step] = correct_labeled_SLP_field

    

:::{note}
In many atmospheric science related features of interest *(ETCs, tropical cyclones, etc.)*, distinct objects may not be located at neighboring grid cells over successive time steps. That is why looping over each time step is **necessary** here.

### Create a map of our objects

Here, we can visualize the objects identified at each time step. The shading on the plot below corresponds to the numerical object identified by the label function.

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.pcolormesh(
        lon2d, lat2d, labeled_arr[t], shading = 'auto',
        cmap=cmap,         
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Identified Low-Pressure Objects | time index = {t}")

In [ ]:
# Create map
vals = labeled_arr
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("Object Number")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

## Find SLP Minima

Here, we utlize scipy's `minimum_filter` function to identify the locations of *local SLP minima*. The minimum_filter function first takes in the n-dimensional array over which we wish to **identify** local minima. Then, `minimum_filter` requires a numerical value that defines the size of the grid over which minima will be found. In the following example, we locate minima within 5x5 grid boxes throughout our domain. The `modes` keyword tells us how to find *minima* at the edges of the domain. In the **y-direction**, we simply use the nearest grid cell within our domain, whereas in the **x-direction** we want to wrap to the other side of our domain since our field is continuous in the x-direction. Lastly, we specify over which axes we wish to calculate minima *(latitude, longitude)*.

:::{warning}
The `minimum_filter` function does not identify minima itself. The two following lines of code in the block below are needed to actually identify the times and locations of **minima**. 
:::

In [ ]:
minima = minimum_filter(SLP_arr, 5, mode=['nearest', 'wrap'], axes = [1, 2])
minima_3 = (SLP_arr == minima)
min_times, min_lats, min_lons = np.where(1 == minima_3)

### Plot SLP and identified minima

In the animation produced by the following block of code, you will see a plot of SLP, with **all identified** local minima shown as <span style="color:blue"> *blue dots* </span>.

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """

    where_time = np.where(min_times == t)

    proper_time_lats = min_lats[where_time]
    proper_time_lons = min_lons[where_time]
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = SLP_hPa.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.contourf(
        lon2d, lat2d, SLP_arr[t],
        cmap=cmap,         
        transform=ccrs.PlateCarree(),
    )

    cs = ax.scatter(
        lon[proper_time_lons], lat[proper_time_lats], 
        zorder = 2,
    )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"SLP and its Minima | time index = {t}")

In [ ]:
# Create map
vals = SLP_arr
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo'

fig, ax = plt.subplots(figsize=(9, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("Object Number")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=SLP_hPa.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())

:::{tip}
In the above animation, there are many minima that are likely not associated with features of interest, such as ETCs. The second input to the `minimum_filter` function can be altered to make more stringent requirements for the identification of a **local minimum**.
:::

## Find main SLP minimum for each feature

Below, we loop through each time step and then loop through each identified feature at that time step. First, we identify the *latitudes and longitudes* of our features of interest at each time step. We then look at each **identified feature** and find the **lowest SLP minimum** that occurs within that feature.

In [ ]:
#Find the main SLP center for each object at each time step
deepest_mins_list = []

for time_step in np.arange(0, len(SLP_arr)):
    where_time = np.where(min_times == time_step)
    proper_time_lats = min_lats[where_time]
    proper_time_lons = min_lons[where_time]
    
    deepest_mins = {}
    
    for label_num in range(1, int(labeled_arr[time_step].max() + 1)):

        mask = (labeled_arr[time_step, proper_time_lats, proper_time_lons] == label_num)

        if(not np.any(mask)):
            continue

        obj_lat = proper_time_lats[mask]
        obj_lon = proper_time_lons[mask]

        obj_slp = SLP_arr[time_step][obj_lat, obj_lon]

        lowest = np.argmin(obj_slp)

        deepest_mins[label_num] = (lat[obj_lat[lowest]], obj_lon[lowest])
    deepest_mins_list.append(deepest_mins.copy())
    deepest_mins.clear()
    

## Track features through time

We now have all *proper minima* at each time step. It is time to stitch together the minima through successive time steps to form tracks. The goal is to take **all identified minima** at the first time step in our dataset and track them through time. We introduce a variable in the block below called `still exists`. As the name suggests, when this variable is `True`, the storm track still exists at a certain time step. If it is `False`, then the track no longer exists at a certain time step.

To track features through time, we take a look at the next time step (t+1) and caculate the **distances** between our *minimum* at time step t and all identified features at time step t+1. We then search for a minimum at t+1 that is within 500 km of our minimum at t. This minimum at t+1 is taken as the position of our feature of interest at the next time step. 

In [ ]:
#We now have all minima at each time step. Now, it's time to stitch the tracks together
#Start with time step 1, check to see what's within 500 km. If none, then the track ends

first_time_step = deepest_mins_list[0]

track_no = 1
for min_point in first_time_step:
    print('----------------------------')
    print('Track Number ' + str(track_no))
    time_ind = 0
    
    #Start with minimum for first object
    min_point_lat = first_time_step[min_point][0]
    min_point_lon = first_time_step[min_point][1]
    print('Time step 1:', min_point_lat, min_point_lon)
    still_exists = True

    while(still_exists):
        time_ind += 1

        if(time_ind > len(deepest_mins_list) - 1):
            break
        
        #Go to next time step
        next_time_step = deepest_mins_list[time_ind]
        
        distances = np.zeros((len(next_time_step.items())))
        
        ind = 0
        for feat in next_time_step.values():
            dist = distance(min_point_lon, min_point_lat, feat[1], feat[0])
            distances[ind] = dist

            ind += 1
        
        min_dist = np.argmin(distances)
        #print(min_dist, next_time_step)
        if(min_dist < 500):
            new_min_point_lat = list(next_time_step.items())[min_dist][1][0]
            new_min_point_lon = list(next_time_step.items())[min_dist][1][1]
            print('Time step ' + str(time_ind + 1) + ':', new_min_point_lat, new_min_point_lon)
        else:
            still_exists = False
    track_no += 1
    